# Part 3: Baseline and model evaluation

Owner: _(name)_

**What this notebook is for.** Put the three models and the baseline side by side on the same test days
and say which is better, by how much, and whether the difference means anything.

**What you need first.** The result files from the three model notebooks, in the `results` folder
(`arima_predictions.csv`, `rnn_predictions.csv`, `lstm_predictions.csv` and their `_meta.json`).
Until they are there, this notebook shows only the baseline, which is fine for a first run.

**What you do.** Run the cells in order, then write the text for section 5 of the report (brief: the
baseline, MAE and RMSE in index points, actual-versus-predicted plots, an error table, residual patterns,
overfitting, computational cost, limitations, and whether any improvement is meaningful). A well-argued
"nothing beats the baseline" is a valid result.

### Setup: run this cell first (nothing to change)

It fetches the repository and the data into this Colab session, loads the shared settings, and prints the
three periods: train, validation, test. The first run takes about a minute.

In [ ]:
# ---- Setup (identical in every notebook). Just run it. ----
import sys, os, pathlib, subprocess, importlib.util

REPO_URL = "https://github.com/Makkaii7/sp500-forecasting.git"
EXTRA_PACKAGES = []      # what this notebook needs that Colab does not have by default

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# 1. In Colab, fetch the repository (once per session). On your own computer this step is skipped.
if IN_COLAB and not pathlib.Path("sp500-forecasting").exists() and not pathlib.Path("src/config.py").exists():
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    public = subprocess.run(["git", "clone", "--quiet", REPO_URL], env=env, capture_output=True).returncode == 0
    if not public:                                   # only if the repository was made private again
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            from getpass import getpass
            token = getpass("The repository is private. Paste a GitHub token: ")
        subprocess.run(["git", "clone", "--quiet", REPO_URL.replace("https://", f"https://{token}@")], env=env, check=True)
if IN_COLAB and pathlib.Path("sp500-forecasting").exists():
    os.chdir("sp500-forecasting")

# 2. Install anything missing (usually only on the first run in Colab).
missing = [p for p in EXTRA_PACKAGES if importlib.util.find_spec(p) is None]
if missing:
    print("installing", ", ".join(missing), "... (about a minute)")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# 3. Find the repository folder and load the shared code (src/config.py and src/common.py).
root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src import config, common

# 4. The data everyone uses, and the three periods in time order.
df = common.load_prices()           # columns: Open, High, Low, Close, Volume, log_return; one row per trading day
y = common.target_series(df)        # the daily log return: what the models work on
train, val, test = common.split(y)  # dates from src/config.py
print("repository:", root)
print("data:", df.index.min().date(), "to", df.index.max().date(), "|", len(df), "trading days")
print(f"train       {train.index.min().date()} to {train.index.max().date()}  ({len(train)} days)  the model learns here")
print(f"validation  {val.index.min().date()} to {val.index.max().date()}  ({len(val)} days)  you compare settings here")
print(f"test        {test.index.min().date()} to {test.index.max().date()}  ({len(test)} days)  touched once, at the very end")
df.tail(3)

## 1. The baseline: tomorrow's close = today's close (just run)

The simplest sensible forecast. **MAE** (mean absolute error) is the average size of the mistakes.
**RMSE** (root mean squared error) is similar but counts big mistakes extra. Both are in index points,
the unit of the S&P 500 itself. The baseline is saved like a model so that it appears in the same table.

In [ ]:
baseline = common.last_value_baseline(test.index, df)      # yesterday's close, for every test day
actual = common.actual_price(test.index, df)               # today's close
# saved like a model so it appears in the same table and plots; predicting "no change" is a zero return
common.save_predictions("baseline_last_value", df, pd.Series(0.0, index=test.index), settings={"rule": "close_t = close_{t-1}"}, train_seconds=0.0)
print("baseline MAE  (index points)", round(common.mae(actual, baseline), 2))
print("baseline RMSE (index points)", round(common.rmse(actual, baseline), 2))

## 2. The error table: every model found in `results`, same test days (just run)

The last column says how much larger or smaller each model's RMSE is than the baseline's. Negative is
better than the baseline.

In [ ]:
score = common.scorecard()
base_rmse = score.loc["baseline_last_value", "RMSE (index points)"]
score["RMSE vs baseline (%)"] = ((score["RMSE (index points)"] / base_rmse - 1) * 100).round(2)
score

## 3. Actual versus predicted, one panel per model (just run)

In [ ]:
import glob
files = sorted(glob.glob(str(root / "results" / "*_predictions.csv")))
fig, axes = plt.subplots(len(files), 1, figsize=(11, 3.2 * len(files)), sharex=True)
for ax, f in zip(np.atleast_1d(axes), files):
    p = pd.read_csv(f, parse_dates=[0], index_col=0)
    p["actual_close"].plot(ax=ax, alpha=0.6, label="actual close")
    p["predicted_close"].plot(ax=ax, alpha=0.9, label="predicted close")
    ax.set_title(pathlib.Path(f).stem.replace("_predictions", "")); ax.set_ylabel("index points"); ax.legend()
plt.tight_layout()

## 4. Residual patterns (just run)

A residual is actual minus predicted. If a model captured everything it could, its residuals look like
noise: no drift over time, and ACF bars inside the shaded band. Bars outside the band mean the model left
a pattern behind.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf
fig, axes = plt.subplots(len(files), 2, figsize=(12, 3 * len(files)))
for row, f in zip(np.atleast_2d(axes), files):
    p = pd.read_csv(f, parse_dates=[0], index_col=0)
    resid = (p["actual_close"] - p["predicted_close"]).dropna()
    resid.plot(ax=row[0], title=pathlib.Path(f).stem.replace("_predictions", "") + ": residuals (index points)")
    plot_acf(resid, lags=20, ax=row[1], title="ACF of residuals")
plt.tight_layout()

## 5. Is any improvement meaningful? (just run, then write)

Two models can differ by a few index points for reasons that are pure chance. Before calling a difference
real, check three things: its size next to the baseline (the % column above), whether it holds in both
halves of the test period (the cell below), and whether the model's residuals show a pattern the
baseline's do not. Write the conclusion in words a reader can check against the table.

In [ ]:
half = len(test) // 2
rows = []
for f in files:
    p = pd.read_csv(f, parse_dates=[0], index_col=0)
    name = pathlib.Path(f).stem.replace("_predictions", "")
    rows.append({"model": name,
                 "RMSE first half": common.rmse(p["actual_close"].iloc[:half], p["predicted_close"].iloc[:half]),
                 "RMSE second half": common.rmse(p["actual_close"].iloc[half:], p["predicted_close"].iloc[half:])})
pd.DataFrame(rows).set_index("model").round(2)

## 6. Computational cost and limitations (write here)

Use the "train seconds" column of the table and the settings in each `<model>_meta.json`. Points to cover:
training time against accuracy; how sensitive the neural networks are to the window length and the
random seed; the fact that there is a single test period; and what a one-day-ahead forecast does and
does not tell you.

_your text_